In [1]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())

Tue Oct  6 19:23:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls -lh "/content/drive/MyDrive/Predicting an A1/"

total 91M
-rw------- 1 root root 91M Oct  6 19:18 EuroSAT_RGB.zip


In [6]:
ZIP_NAME = "EuroSAT_RGB.zip"
!cp "/content/drive/MyDrive/Predicting an A1/{ZIP_NAME}" /content/
!unzip -q "/content/{ZIP_NAME}" -d /content/eurosat
!ls /content/eurosat

EuroSAT_RGB


In [7]:
from pathlib import Path

# find the folder that directly contains the 10 class folders
root = next(p.parent for p in Path("/content/eurosat").rglob("Forest") if p.is_dir())
print("Image root:", root)

total = 0
for d in sorted(root.iterdir()):
    if d.is_dir():
        n = len(list(d.glob("*.jpg")))
        total += n
        print(f"{d.name:22s} {n}")
print("TOTAL:", total)

Image root: /content/eurosat/EuroSAT_RGB
AnnualCrop             3000
Forest                 3000
HerbaceousVegetation   3000
Highway                2500
Industrial             2500
Pasture                2000
PermanentCrop          2500
Residential            3000
River                  2500
SeaLake                3000
TOTAL: 27000


In [8]:
!pip install -q transformers
import transformers, torchvision, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| transformers", transformers.__version__)

python 3.13.15
torch 2.11.0+cu130 | torchvision 0.26.0+cu130 | transformers 5.18.0


In [9]:
# ---- Cell 7: extract frozen ViT fingerprints (CLS token, no augmentation) ----
import os
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTModel

ROOT = "/content/eurosat/EuroSAT_RGB"
OUT  = "/content/drive/MyDrive/aai-project/features"
CHECKPOINT = "google/vit-base-patch16-224-in21k"
BATCH = 256          # drop to 128 if you run out of memory
os.makedirs(OUT, exist_ok=True)

# Same preprocessing as the plan's eval transform: bicubic 64->224, normalise mean/std 0.5
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

class EuroSATFolder(Dataset):
    def __init__(self, root):
        self.items = []
        for class_dir in sorted(Path(root).iterdir()):
            if class_dir.is_dir():
                for p in sorted(class_dir.glob("*.jpg")):
                    self.items.append((f"{class_dir.name}/{p.stem}", class_dir.name, str(p)))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        return TRANSFORM(Image.open(self.items[i][2]).convert("RGB"))

ds = EuroSATFolder(ROOT)
print("images found:", len(ds))
loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

device = "cuda"
model = ViTModel.from_pretrained(CHECKPOINT, add_pooling_layer=False).to(device).eval()

feats = []
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    for i, x in enumerate(loader):
        out = model(pixel_values=x.to(device, non_blocking=True))
        feats.append(out.last_hidden_state[:, 0].float().cpu())   # CLS after final layernorm
        if i % 20 == 0:
            print(f"batch {i}/{len(loader)}")

feats = torch.cat(feats).numpy()
assert feats.shape == (len(ds), 768), feats.shape

np.save(f"{OUT}/fingerprints_view0.npy", feats.astype(np.float16))
pd.DataFrame(ds.items, columns=["key", "label", "path"]).to_csv(f"{OUT}/fingerprints_index.csv", index=False)

print("saved", feats.shape)
print(pd.Series([it[1] for it in ds.items]).value_counts().sort_index())
print("any NaN:", np.isnan(feats).any())

images found: 27000


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


batch 0/106
batch 20/106
batch 40/106
batch 60/106
batch 80/106
batch 100/106
saved (27000, 768)
AnnualCrop              3000
Forest                  3000
HerbaceousVegetation    3000
Highway                 2500
Industrial              2500
Pasture                 2000
PermanentCrop           2500
Residential             3000
River                   2500
SeaLake                 3000
Name: count, dtype: int64
any NaN: False


In [10]:
OUT = "/content/drive/MyDrive/aai-project/features"
!ls -lh "{OUT}"

total 43M
-rw------- 1 root root 2.5M Oct  6 19:32 fingerprints_index.csv
-rw------- 1 root root  40M Oct  6 19:32 fingerprints_view0.npy


In [11]:
import numpy as np, pandas as pd

X = np.load(f"{OUT}/fingerprints_view0.npy").astype(np.float32)
idx = pd.read_csv(f"{OUT}/fingerprints_index.csv")
X /= np.linalg.norm(X, axis=1, keepdims=True)   # normalise so dot product = cosine similarity

rng = np.random.default_rng(0)
def mean_sim(a, b, n=2000):
    ia = rng.choice(np.where(idx.label == a)[0], n)
    ib = rng.choice(np.where(idx.label == b)[0], n)
    return float((X[ia] * X[ib]).sum(1).mean())

print("Forest vs Forest        :", round(mean_sim("Forest", "Forest"), 3))
print("Forest vs Highway       :", round(mean_sim("Forest", "Highway"), 3))
print("SeaLake vs SeaLake      :", round(mean_sim("SeaLake", "SeaLake"), 3))
print("SeaLake vs Industrial   :", round(mean_sim("SeaLake", "Industrial"), 3))

Forest vs Forest        : 0.452
Forest vs Highway       : 0.189
SeaLake vs SeaLake      : 0.41
SeaLake vs Industrial   : 0.093
